# __ASE UMA implementation__
This notebook contains the installation procedure and initial testing provided in the UMA [HuggingFace](https://huggingface.co/facebook/UMA) (pseudo-GitHub repository), which also contains the Meta paper for these models. These contain the weights (checkpoint files) that are used by the ASE calculator via the `FAIR` python package, also by Meta ([GitHub](https://github.com/facebookresearch/fairchem))

**Obs.** The provided calculator requires the weights as well as the task, which is detailed in the `FAIR` [repo](https://github.com/facebookresearch/fairchem). 

Its dataset, OMol25, can be found in its [HuggingFace repo](https://huggingface.co/facebook/OMol25). 

### __Installation procedure__
Refer to ASE installation for the basics of the miniconda `venv` where UMA will be installed. This ensures that other calculators will not break the environment.
- ASE installation process
    - Create new conda environment (python 3.10 is standard). Check the Python requirement for the external calculator wanted. Usually  `python>3.9`.  
    - `conda install ase` (installs `scipy` and `numpy` dependencies) and `conda install matplotlib`. 
    - Check the environment works installing `conda install pytest` and `ase test`.
    - For ASE representations inside Jupyter Notebook, conda install `notebook`, `ipywidgets` and `nglview`. 
- UMA installation process
    - Clone base ASE environment. 
    - Request access and download the weights (checkpoint files) [here](https://huggingface.co/facebook/UMA). 
    - Install `FAIR` i.e. the `ASE` calculator for UMA models with `conda install fairchem-core`.  
    - (if it does not work, `conda uninstall fairchem-core` and use `pip install fairchem-core`). 
    - `FAIR` contains eSEN and UMA models as 'templates', to use them only the optimized weights are required. 

### __References__
- How to get started with UMA models. Includes setup and some examples
    - https://rowansci.com/blog/how-to-run-open-molecules-2025

- FAIR GitHub. Provides some initial setup tests to check everything works and the syntax to the calculators. 
    - https://github.com/facebookresearch/fairchem

## __UMA pretrained models__
UMA models are obtained from `FAIR`, a Meta repo that contains the 'templates' for UMA and eSEN (from which UMA models are partly based on) models. These are then casted to ASE calculators names `FAIRChemCalculator` from the weights (checkpoint files). 

The pretrained models available are as follows:
- `uma-s-1p1`
    - Latest version of the UMA small model, fastest of the UMA models while still SOTA on most benchmarks (6.6M/150M active/total params)


- `uma-m-1p1`
    - Best in class UMA model across all metrics, but slower and more memory intensive than uma-s (50M/1.4B active/total params)

Further information on the dataset and data details can be found in the [demo](https://huggingface.co/spaces/facebook/fairchem_uma_demo)

The tasks available are as follows
- `oc20` for catalysis
- `omat` for inorganic materials
- `omol` for molecules and metal-organic complexes. Trained on OMol25. 
- `odac` for MOFs
- `omc` for molecular crystals

**Note.** Currently, all UMA models on `omol` mode require `charge` and `spin` (`spin_multiplicity`) that must be specified in `atoms.info` during calculations. If not set, `charge` and `spin` will default to 0 and 1, respectively (neutral singlet). 

In [ ]:
from fairchem.core import FAIRChemCalculator
from fairchem.core.units.mlip_unit import load_predict_unit

# ESEN
# note depending on the eSEN model forces are predicted directly or from - grad E
# create the eSEN model from a template and the weights
esen_predictor = load_predict_unit(
    path="esen_sm_conserving_all.pt",
    device="cpu",
)

# cast it to an ASE calculator
esen_calculator = FAIRChemCalculator(esen_predictor)


# UMA
# create the UMA model from a template and the weights
uma_predictor = load_predict_unit(
    path='uma-s-1.pt', 
    device='cpu',                   # defaults to 'cuda' 
    inference_settings='default',   # 'turbo' option available
)

# cast it to an ASE calculator
uma_calculator = FAIRChemCalculator(
    uma_predictor,
    task_name='omol',               # other tasks available
)


## __1. ASE UMA calculator usage__

### __H2O single point calculation and atomization energy__

In [ ]:
# ASE system
from ase import build
atoms = build.molecule('H2O')


from fairchem.core import FAIRChemCalculator
from fairchem.core.units.mlip_unit import load_predict_unit

# define the UMA ASE calculator
uma_predictor = load_predict_unit(
    path='uma-s-1p1.pt', 
    device='cpu',                   
    inference_settings='default',   
)
uma_calc = FAIRChemCalculator(
    uma_predictor,
    task_name='omol',               
)


atoms.calc = uma_calc
atoms.info = {"charge": 0, "spin": 1}           # ONLY REQUIRED IN UMA
uma_H2O_energy = atoms.get_potential_energy()
print(f'H2O energy @ UMA (small, OMol25):\t{uma_H2O_energy}')

/Users/sergiortizropero/miniconda3/envs/ASE_UMA_test/lib/python3.12/site-packages/torchtnt/utils/version.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
W0707 22:15:47.666000 46184 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


H2O energy @ UMA (small, OMol25):	-2079.8623617269677


In [ ]:
# check the energy of formation of the H2O molecule
calc_list = [uma_calc]
atomic_energies = []

k = 0
for calc_ in calc_list:
    # define the atoms
    atom_O = build.molecule('O')
    atom_O.info = {"charge": 0, "spin": 3}
    atom_H = build.molecule('H')
    atom_H.info = {"charge": 0, "spin": 2}

    atom_O.calc = calc_
    atom_H.calc = calc_

    energy_H = atom_H.get_potential_energy()
    energy_O = atom_O.get_potential_energy()
    atomic_energies.append([energy_H, energy_O])

    k += 0

# check the results
delta_E_formation_UMA = uma_H2O_energy - (2 * atomic_energies[0][0] + atomic_energies[0][1])


print(atomic_energies)
print(f'H2O energy formation @ UMA (small, OMol25):\t{delta_E_formation_UMA:.3f} [eV]')

### __H2O MD NVT simulation__
This simulation ([reference](https://mace-docs.readthedocs.io/en/latest/examples/foundation_examples.html)) uses the UMA foundational model as NNP to compute forces. 

In [ ]:
from ase import build
from ase.md import Langevin
from ase.io.trajectory import Trajectory
from ase.visualize import view
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase import units

# MD settings
T_init = 300 
n_steps = 200


atoms = build.molecule('H2O')
atoms.info = {"charge": 0, "spin": 1}
atoms.calc = uma_calc

MaxwellBoltzmannDistribution(atoms, temperature_K=T_init*units.kB)

traj = Trajectory('water_nvt.traj', 'w', atoms)

# NVT
dyn = Langevin(atoms, 0.5 * units.fs, temperature_K=T_init*units.kB, friction=0.001)
dyn.attach(traj.write, interval=1)
dyn.run(n_steps)

In [ ]:
traj = Trajectory('water_nvt.traj')
view(traj, viewer='ngl')

## __2. UMA Architecture__